# Analog Computing Module
## Continuous Dynamical Systems and Hybrid Computation

This notebook covers:

1. ODE Solving with Analog Principles
2. Harmonic Oscillator Networks
3. Optimization via Continuous Dynamics
4. Gradient Descent as Dynamical System
5. Hopfield Network Energy Landscape

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp, odeint
from scipy.optimize import minimize

print("Analog Computing Module Loaded")
print("=" * 50)

## 1. ODE Solving - The Core of Analog Computing

Analog computers solve differential equations by representing variables as continuous voltages/currents.

In [ ]:
# Lotka-Volterra (Predator-Prey) System
# Analog computers naturally model these continuous dynamics

def lotka_volterra(t, y, alpha=1.5, beta=1.0, delta=1.0, gamma=1.0):
    """
    Lotka-Volterra predator-prey equations
    dx/dt = alpha*x - beta*x*y  (prey)
    dy/dt = delta*x*y - gamma*y  (predator)
    """
    x, y = y
    dxdt = alpha * x - beta * x * y
    dydt = delta * x * y - gamma * y
    return [dxdt, dydt]

# Solve using scipy (digital simulation of analog process)
t_span = (0, 50)
y0 = [2, 1]  # Initial prey and predator populations

t = np.linspace(0, 50, 1000)
solution = solve_ivp(lotka_volterra, t_span, y0, t_eval=t)

# Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax1 = axes[0]
ax1.plot(solution.t, solution.y[0], 'b-', label='Prey (x)', linewidth=2)
ax1.plot(solution.t, solution.y[1], 'r-', label='Predator (y)', linewidth=2)
ax1.set_xlabel('Time')
ax1.set_ylabel('Population')
ax1.set_title('Lotka-Volterra Oscillations')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2 = axes[1]
ax2.plot(solution.y[0], solution.y[1], 'purple', linewidth=1)
ax2.scatter([y0[0]], [y0[1]], c='green', s=100, zorder=5, label='Start')
ax2.scatter([solution.y[0, -1]], [solution.y[1, -1]], c='red', s=100, zorder=5, label='End')
ax2.set_xlabel('Prey (x)')
ax2.set_ylabel('Predator (y)')
ax2.set_title('Phase Portrait')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 2. Harmonic Oscillator Networks

Analog computers excel at modeling coupled harmonic oscillators, which appear in molecular dynamics and control systems.

In [ ]:
# Coupled Harmonic Oscillators

def coupled_oscillators(t, y, n_oscillators=3, k=1.0, coupling=0.5):
    """
    n coupled harmonic oscillators with spring coupling
    """
    n = n_oscillators
    positions = y[:n]
    velocities = y[n:]
    
    # Acceleration for each oscillator
    accelerations = np.zeros(n)
    
    for i in range(n):
        # Self spring (to equilibrium)
        acc = -k * positions[i]
        
        # Coupling to neighbors
        if i > 0:
            acc += coupling * (positions[i-1] - positions[i])
        if i < n-1:
            acc += coupling * (positions[i+1] - positions[i])
        
        accelerations[i] = acc
    
    return np.concatenate([velocities, accelerations])

# Simulate
n = 5
y0 = np.concatenate([np.sin(np.linspace(0, 2*np.pi, n)), np.zeros(n)])
y0[n//2] = 1  # Initial displacement at center

t = np.linspace(0, 50, 2000)
sol = solve_ivp(coupled_oscillators, (0, 50), y0, t_eval=t)

# Plot
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8))

for i in range(n):
    ax1.plot(sol.t, sol.y[i], alpha=0.7, label=f'Osc {i+1}')

ax1.set_xlabel('Time')
ax1.set_ylabel('Displacement')
ax1.set_title('Coupled Harmonic Oscillators')
ax1.legend(ncol=5)
ax1.grid(True, alpha=0.3)

# Phase space
for i in range(n):
    ax2.plot(sol.y[i], sol.y[i+n], alpha=0.5)
ax2.set_xlabel('Position')
ax2.set_ylabel('Velocity')
ax2.set_title('Phase Space Trajectories')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 3. Optimization via Continuous Dynamics

Analog computers naturally perform optimization through energy minimization in physical systems.

In [ ]:
# Simulated Annealing as Continuous Dynamical System

def rastrigin(x):
    """Rastrigin function - classic optimization benchmark"""
    A = 10
    return A * len(x) + np.sum(x**2 - A * np.cos(2 * np.pi * x))

def gradient_descent_dynamics(x, lr=0.01, momentum=0.9):
    """
    Simulate gradient descent as a dynamical system
    """
    trajectory = [x.copy()]
    velocity = np.zeros_like(x)
    
    for _ in range(1000):
        # Compute gradient (central differences)
        grad = np.zeros_like(x)
        eps = 1e-8
        for i in range(len(x)):
            x_plus = x.copy()
            x_plus[i] += eps
            x_minus = x.copy()
            x_minus[i] -= eps
            grad[i] = (rastrigin(x_plus) - rastrigin(x_minus)) / (2 * eps)
        
        # Update with momentum
        velocity = momentum * velocity - lr * grad
        x = x + velocity
        trajectory.append(x.copy())
    
    return np.array(trajectory)

# Run from multiple starting points
np.random.seed(42)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax1, ax2 = axes

# 2D Rastrigin landscape
x = np.linspace(-5, 5, 200)
y = np.linspace(-5, 5, 200)
X, Y = np.meshgrid(x, y)
Z = np.array([[rastrigin([xi, yi]) for xi, yi in zip(x_row, y_row)] 
              for x_row, y_row in zip(X, Y)])

ax1.contourf(X, Y, Z, levels=50, cmap='viridis')
ax1.contour(X, Y, Z, levels=20, colors='white', alpha=0.3)

# Run gradient descent from different points
starting_points = [[4, 4], [-4, 4], [4, -4], [-4, -4], [0, 4]]
colors = plt.cm.rainbow(np.linspace(0, 1, len(starting_points)))

for start, color in zip(starting_points, colors):
    traj = gradient_descent_dynamics(np.array(start))
    ax1.plot(traj[:, 0], traj[:, 1], c=color, linewidth=1, alpha=0.8)
    ax1.scatter(traj[0, 0], traj[0, 1], c=[color], s=50, marker='o')
    ax1.scatter(traj[-1, 0], traj[-1, 1], c=[color], s=50, marker='*')

ax1.scatter(0, 0, c='white', s=200, marker='X', edgecolor='black', label='Global Minimum')
ax1.set_xlabel('x')
ax1.set_ylabel('y')
ax1.set_title('Rastrigin Function with Gradient Descent Trajectories')
ax1.legend()

# Convergence plot
traj = gradient_descent_dynamics(np.array([4, 4]))
energies = [rastrigin(x) for x in traj]
ax2.plot(energies, 'b-', linewidth=1)
ax2.set_xlabel('Iteration')
ax2.set_ylabel('Objective Value')
ax2.set_title('Optimization Convergence')
ax2.set_yscale('log')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Final value: {energies[-1]:.4f}")
print(f"Global minimum: 0.0")

## 4. Gradient Descent as Dynamical System

The continuous limit of gradient descent is a dynamical system that naturally evolves toward minima.

In [ ]:
# Continuous Gradient Flow

def gradient_flow(t, y, objective_func, grad_func=None):
    """
    Continuous gradient flow: dy/dt = -∇f(y)
    """
    if grad_func is None:
        # Numerical gradient
        eps = 1e-8
        grad = np.zeros_like(y)
        for i in range(len(y)):
            y_plus = y.copy()
            y_plus[i] += eps
            y_minus = y.copy()
            y_minus[i] -= eps
            grad[i] = (objective_func(y_plus) - objective_func(y_minus)) / (2 * eps)
    else:
        grad = grad_func(y)
    
    return -grad

# Example: Rosenbrock function
def rosenbrock(x):
    return (1 - x[0])**2 + 100 * (x[1] - x[0]**2)**2

def rosenbrock_grad(x):
    dx0 = -2 * (1 - x[0]) - 400 * x[0] * (x[1] - x[0]**2)
    dx1 = 200 * (x[1] - x[0]**2)
    return np.array([dx0, dx1])

# Solve gradient flow
y0 = np.array([-1.0, 1.0])
t_flow = np.linspace(0, 5, 500)

sol_flow = solve_ivp(
    lambda t, y: gradient_flow(t, y, rosenbrock, rosenbrock_grad),
    (0, 5), y0, t_eval=t_flow
)

# Plot
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Contour plot
ax1 = axes[0]
x = np.linspace(-2, 2, 200)
y = np.linspace(-1, 3, 200)
X, Y = np.meshgrid(x, y)
Z = np.array([[rosenbrock([xi, yi]) for xi in x] for yi in y])

ax1.contourf(X, Y, Z, levels=50, cmap='viridis')
ax1.contour(X, Y, Z, levels=np.logspace(-1, 3, 20), colors='white', alpha=0.5)
ax1.plot(sol_flow.y[0], sol_flow.y[1], 'r-', linewidth=2, label='Gradient Flow')
ax1.scatter(sol_flow.y[0, 0], sol_flow.y[1, 0], c='green', s=100, zorder=5, label='Start')
ax1.scatter(1, 1, c='white', s=100, marker='X', zorder=5, label='Minimum')
ax1.set_xlabel('x')
ax1.set_ylabel('y')
ax1.set_title('Gradient Flow on Rosenbrock')
ax1.legend()

# Trajectories in 3D
ax2 = fig.add_subplot(122, projection='3d')
ax2.plot(sol_flow.y[0], sol_flow.y[1], t_flow)
ax2.set_xlabel('x')
ax2.set_ylabel('y')
ax2.set_zlabel('Time')
ax2.set_title('Trajectory in (x, y, t) Space')

# Objective value
ax3 = axes[2]
values = [rosenbrock(sol_flow.y[:, i]) for i in range(len(t_flow))]
ax3.semilogy(t_flow, values, 'b-')
ax3.set_xlabel('Time')
ax3.set_ylabel('Rosenbrock(x, y)')
ax3.set_title('Objective Evolution')
ax3.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 5. Hopfield Network Energy Landscape

Hopfield networks are recurrent neural networks that can store patterns as stable states in an energy landscape.

In [ ]:
class HopfieldNetwork:
    """
    Hopfield Network for associative memory
    """
    def __init__(self, n_neurons):
        self.n = n_neurons
        self.W = np.zeros((n_neurons, n_neurons))
        self.threshold = 0
    
    def store_patterns(self, patterns):
        """Store patterns using Hebbian learning"""
        n_patterns = len(patterns)
        for p in patterns:
            self.W += np.outer(p, p)
        self.W = self.W / self.n
        np.fill_diagonal(self.W, 0)  # No self-connections
    
    def energy(self, state):
        """Compute network energy"""
        return -0.5 * state @ self.W @ state + np.sum(state * self.threshold)
    
    def recall(self, pattern, max_iter=100):
        """Recall stored pattern from noisy input"""
        state = pattern.copy()
        energies = [self.energy(state)]
        
        for _ in range(max_iter):
            new_state = np.sign(self.W @ state - self.threshold)
            new_state[new_state == 0] = state[new_state == 0]  # Keep zero as zero
            
            if np.array_equal(new_state, state):
                break
            state = new_state
            energies.append(self.energy(state))
        
        return state, energies

# Create Hopfield network
np.random.seed(42)
n = 100
n_patterns = 5

# Generate random orthogonal-ish patterns
patterns = []
for _ in range(n_patterns):
    p = np.random.choice([-1, 1], size=n)
    patterns.append(p)

network = HopfieldNetwork(n)
network.store_patterns(patterns)

# Test recall with noise
test_pattern = patterns[0].copy()
noise_level = 0.2
noisy_pattern = test_pattern.copy()
flip_indices = np.random.choice(n, int(n * noise_level), replace=False)
noisy_pattern[flip_indices] *= -1

# Recall
recalled, energies = network.recall(noisy_pattern)

# Accuracy
accuracy = np.mean(recalled == test_pattern)

print(f"Pattern recall accuracy: {accuracy*100:.1f}%")
print(f"Energy before recall: {energies[0]:.2f}")
print(f"Energy after recall: {energies[-1]:.2f}")

In [ ]:
# Visualize Hopfield network

fig, axes = plt.subplots(2, 3, figsize=(15, 10))

# Original patterns
for i in range(3):
    ax = axes[0, i]
    ax.imshow(patterns[i].reshape(10, 10), cmap='RdBu', vmin=-1, vmax=1)
    ax.set_title(f'Stored Pattern {i+1}')
    ax.axis('off')

# Noisy input and recall
axes[1, 0].imshow(noisy_pattern.reshape(10, 10), cmap='RdBu', vmin=-1, vmax=1)
axes[1, 0].set_title('Noisy Input')
axes[1, 0].axis('off')

axes[1, 1].imshow(recalled.reshape(10, 10), cmap='RdBu', vmin=-1, vmax=1)
axes[1, 1].set_title('Recalled Pattern')
axes[1, 1].axis('off')

axes[1, 2].plot(energies, 'b-', linewidth=2)
axes[1, 2].set_xlabel('Iteration')
axes[1, 2].set_ylabel('Energy')
axes[1, 2].set_title('Energy Minimization')
axes[1, 2].grid(True, alpha=0.3)

plt.suptitle('Hopfield Network Associative Memory', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Summary

print("\n" + "=" * 60)
print("Analog Computing Module Summary")
print("=" * 60)

print("\nKey Concepts:")
print("  • ODE solvers model continuous physical systems")
print("  • Harmonic oscillators encode periodic dynamics")
print("  • Physical systems naturally minimize energy")
print("  • Gradient flow converges to local minima")
print("  • Hopfield networks store patterns as attractors")

print("\nHardware Implementations:")
print("  • FPAA: Field Programmable Analog Arrays")
print("  • OTA-C: Operational Transconductance Amplifier circuits")
print("  • Photonic: Optical analog processors")
print("  • Memristor: Analog memory devices")